# 05 · Training su ASVspoof 5 intero (Colab)

Allena su Google Colab con il **train intero di ASVspoof 5** (circa 180.000 clip), con la stessa ricetta
degli esperimenti su 2019 LA: RawBoost algoritmo 5, seed 1. L'epoca migliore si sceglie sul solito dev da 5.000 clip.
Di default allena solo MamBo; altri modelli si aggiungono nella cella 3, se restano crediti.
I test (ASVspoof 5 eval completo, 2021 LA, In-the-Wild...) si fanno poi sul fisso, dove ci sono già i dati.

**Prima di partire**
- GPU **A100** (o L4). La T4 non va bene: non supporta bene il bfloat16 che usiamo.
  In VS Code si sceglie quando crei il server (Select Kernel → Colab → New Colab Server); nel browser da Runtime → Cambia tipo di runtime.
- Serve il token di Hugging Face (lo stesso del fisso), perché ASVspoof 5 si scarica da lì.
- L'audio si scarica a ogni sessione sul disco di Colab (circa 57 GB, i TAR si cancellano dopo l'estrazione).
- Checkpoint e log vanno su Drive (`MyDrive/tesi/artifacts`), così non si perdono se Colab si disconnette.
  Se succede, basta rilanciare il notebook dall'inizio: riparte dall'ultima epoca finita.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/tesi"
os.makedirs(DRIVE, exist_ok=True)
os.environ["THESIS_DATA_ROOT"] = "/content/data"            # audio (disco di Colab)
os.environ["THESIS_ARTIFACTS_ROOT"] = DRIVE + "/artifacts"   # checkpoint e log (Drive)
!nvidia-smi --query-gpu=name,memory.total --format=csv
print("CPU:", os.cpu_count())

## 1. Codice

Scarica il repo e installa le librerie. PyTorch su Colab c'è già.

In [ ]:
RAMO = "claude/project-thread-awrzvp"
if not os.path.exists("/content/SB2526-MasterThesis"):
    !git clone -b {RAMO} https://github.com/espositic/SB2526-MasterThesis /content/SB2526-MasterThesis
%cd /content/SB2526-MasterThesis
!git pull
!pip install -q transformers huggingface_hub soundfile imageio-ffmpeg

## 2. Dati

Login a Hugging Face (incolla il token quando lo chiede), poi download ed estrazione di train e dev di ASVspoof 5.
Si fa a ogni nuova sessione: da Hugging Face a Colab è veloce.

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from tesi import dati
for split in ("train", "dev"):
    dati.scarica_asvspoof5(split, cancella_tar=True)
!df -h /content | tail -1

## 3. Training

Lancia la coda. L'output va anche in `MyDrive/tesi/coda_colab.log`: lì guardi i minuti per epoca e l'EER sul dev.
Per allenare anche altri modelli dopo MamBo, aggiungili al comando, per esempio `colab mambo nes2net`.

In [ ]:
!python coda_tabella3.py colab mambo 2>&1 | tee -a {DRIVE}/coda_colab.log

## 4. Portare il modello sul fisso

Copia da Drive la cartella `artifacts/esperimenti/mambo_asvspoof5intero_rawboost/seed1/` (basta `migliore.pt`, `info.json`,
`log.csv` e `completato.json`) in `artifacts\esperimenti\` sul fisso. Lì si fanno i test con il notebook 03.

In [ ]:
import json
from tesi import percorsi
for cartella in sorted(percorsi.ESPERIMENTI.glob("*_asvspoof5intero_*/seed1")):
    fatto = cartella / "completato.json"
    print(cartella.parent.name, json.loads(fatto.read_text()) if fatto.exists() else "in corso")
    if (cartella / "log.csv").exists():
        print(open(cartella / "log.csv").read())